# VibeML: Evaluation Traces

This notebook runs the VibeML agent through five trace scenarios and evaluates the results using MLflow. One trace runs the same scenario through two different LLMs to compare performance. Two traces test how the agent handles requests that fall outside its scope. A custom judge is used to score the agent's recommendations, and commentary is provided on what the evaluation showed.

## Setup

This pulls in the tools and agent logic built in the previous notebook and sets up an MLflow experiment to track all five traces.


In [0]:
%pip install --upgrade "mlflow[databricks]>=3.4.0"
dbutils.library.restartPython()

In [0]:
import mlflow
import json
import pandas as pd

# Set up the MLflow experiment for this notebook
mlflow.set_experiment("/Shared/VibeML_Evaluation_Traces")

print("MLflow experiment set")

## Bringing in the Agent

This pulls in everything from the agent prototype notebook, including the tools, the Spotify connection, the LLM setup, and the datasets, so we can run the agent here without rebuilding it.

In [0]:
%run "./02_agent_prototype"

## MLflow Tracing

Each trace scenario is wrapped in @mlflow.trace so it gets a real trace_id, viewable in the MLflow Traces UI. The judge is built using make_judge(), and both the judge's score and the human review are logged as Feedback directly on each trace.

#### Shared Setup for Trace Scenarios

This section wraps all 8 tools with tracing, defines all 5 user requests used across the trace scenarios, and defines run_rejection_trace, the shared function used by Trace 4 and 5. run_vibeml_trace (used by Trace 1, 2, and 3) is defined separately right after this.

In [0]:
import mlflow

# Wrap each tool so tool calls appear as spans inside each request trace
get_user_profile = mlflow.trace(name="get_user_profile")(get_user_profile)
search_songs = mlflow.trace(name="search_songs")(search_songs)
vibe_mapping = mlflow.trace(name="vibe_mapping")(vibe_mapping)
feedback_refinement = mlflow.trace(name="feedback_refinement")(feedback_refinement)
constraint_filter = mlflow.trace(name="constraint_filter")(constraint_filter)
deduplication = mlflow.trace(name="deduplication")(deduplication)
search_catalog = mlflow.trace(name="search_catalog")(search_catalog)
liked_songs_matcher = mlflow.trace(name="liked_songs_matcher")(liked_songs_matcher)

print("All 8 tools wrapped with tracing")

In [0]:
user_request_1 = "I want fast afrobeat music for the gym"
user_request_2 = "I'm going on a late night drive, give me something that matches that vibe"
user_request_3 = "I need to focus, I'm studying for an exam. Something calm with no lyrics please"
user_request_4 = "Can you recommend a good podcast to listen to on my commute?"
user_request_5 = "Hey, what stocks should I invest in right now?"

print("All 5 user requests defined")

In [0]:
@mlflow.trace(name="vibeml_recommendation")
def run_vibeml_trace(user_request, llm_model_override=None):
    """
    Runs one full recommendation cycle, now exercising all 6 per-request tools:
    vibe_mapping, search_songs, liked_songs_matcher, constraint_filter,
    deduplication, and search_catalog.
    """
    global LLM_MODEL
    original_model = LLM_MODEL
    if llm_model_override:
        LLM_MODEL = llm_model_override

    ranges = vibe_mapping(user_request, profile)
    explicit_genre = extract_explicit_genre(user_request)
    if explicit_genre:
        ranges["genre"] = explicit_genre

    results = search_songs(
        energy=(ranges["energy"]["min"], ranges["energy"]["max"]),
        valence=(ranges["valence"]["min"], ranges["valence"]["max"]),
        danceability=(ranges["danceability"]["min"], ranges["danceability"]["max"]),
        genre=ranges.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results = constraint_filter(results, explicit=False, min_popularity=30)
    results = deduplication(results, max_per_artist=1)

    liked_matches = liked_songs_matcher(profile, ranges, df_tracks)
    if len(liked_matches) > 0:
        liked_matches["is_familiar"] = True
        results = pd.concat([liked_matches, results]).drop_duplicates(subset=["track_name"]).reset_index(drop=True)

    if llm_model_override:
        LLM_MODEL = original_model

    top_results = results.head(5)

    verified_songs = []
    for _, row in top_results.iterrows():
        catalog_result = search_catalog(sp, track_name=row["track_name"], artist_name=row["artists"].split(";")[0])
        verified_songs.append({
            "track_name": row["track_name"],
            "artists": row["artists"],
            "track_genre": row["track_genre"],
            "verified_on_spotify": catalog_result is not None
        })

    return {
        "genre": ranges.get("genre"),
        "songs_found": len(results),
        "top_songs": verified_songs
    }

print("Updated traced function defined")

In [0]:
@mlflow.trace(name="vibeml_rejection")
def run_rejection_trace(user_request):
    """
    Tests whether the agent gracefully declines an out-of-scope request.
    """
    rejection_prompt = f"""You are VibeML, a personalized Spotify playlist agent. You only help with music playlist requests.
    
    A user just said: "{user_request}"
    
    Respond naturally, explaining that this falls outside what you can help with since you are scoped to music playlists, 
    and then redirect them back to building a playlist."""
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": rejection_prompt}]
    )
    
    content = response.choices[0].message.content
    if isinstance(content, list):
        rejection_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        rejection_text = content
    
    return rejection_text

#### Separate Trace: User Profile Pull

get_user_profile only runs once per session in the real agent, not once per request, so it gets its own standalone trace rather than being inside every scenario.

In [0]:
@mlflow.trace(name="vibeml_profile_pull")
def run_profile_trace():
    return get_user_profile(sp)

profile_result = run_profile_trace()
profile_trace_id = mlflow.get_last_active_trace_id()

print(f"Profile trace logged: {profile_trace_id}")
print(f"Top artists: {profile_result['top_artists'][:5]}")
print(f"Preferred genres: {profile_result['preferred_genres']}")

##### Commentary

#### Trace 1: Gym / High Energy (Dual LLM Comparison)

This trace runs the same user request through two different LLMs, databricks-gpt-oss-120b and databricks-meta-llama-3-3-70b-instruct, so we can compare how each one interprets the mood and picks songs.

In [0]:
trace1_result = run_vibeml_trace(user_request_1, llm_model_override="databricks-gpt-oss-120b")
trace1_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace1_id}")
print(f"Genre: {trace1_result['genre']}")
print(f"Songs found: {trace1_result['songs_found']}")
for song in trace1_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

In [0]:
trace1b_result = run_vibeml_trace(user_request_1, llm_model_override="databricks-meta-llama-3-3-70b-instruct")
trace1b_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace1b_id}")
print(f"Genre: {trace1b_result['genre']}")
print(f"Songs found: {trace1b_result['songs_found']}")
for song in trace1b_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

##### Commentary

#### Trace 2: Late Night Drive

This trace tests a more abstract, mood-based request rather than a direct genre request. The user describes a feeling rather than naming a genre, so this checks how well the agent translates context into audio features without an explicit genre override.

In [0]:
trace2_result = run_vibeml_trace(user_request_2)
trace2_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace2_id}")
print(f"Genre: {trace2_result['genre']}")
print(f"Songs found: {trace2_result['songs_found']}")
for song in trace2_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

##### Commentary

#### Trace 3: Studying / No Lyrics

This trace tests a functional request rather than a mood-based one. The user wants something specific for focus, low energy, no lyrics, which checks whether the agent correctly prioritizes low speechiness and high instrumentalness instead of just defaulting to a generic "calm" genre.

In [0]:
trace3_result = run_vibeml_trace(user_request_3)
trace3_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace3_id}")
print(f"Genre: {trace3_result['genre']}")
print(f"Songs found: {trace3_result['songs_found']}")
for song in trace3_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

##### Commentary

#### Trace 4: Graceful Rejection 1 — Podcast Request

This trace tests whether the agent correctly recognizes a request that falls outside its scope and declines it cleanly instead of trying to force a music-based answer or hallucinating a response.

In [0]:
rejection_text_4 = run_rejection_trace(user_request_4)
trace4_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace4_id}")
print(f"User: {user_request_4}")
print(f"\nVibeML: {rejection_text_4}")

##### Commentary

#### Trace 5: Graceful Rejection 2 — Financial Advice Request

This trace tests a different kind of out-of-scope request, one that is not music-adjacent at all. This checks whether the agent holds its boundaries consistently across different types of irrelevant requests, not just ones that are loosely related to audio content like the podcast example.

In [0]:
rejection_text_5 = run_rejection_trace(user_request_5)
trace5_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace5_id}")
print(f"User: {user_request_5}")
print(f"\nVibeML: {rejection_text_5}")

##### Commentary

## Judge Evaluation

Runs vibe_match_judge against each of the four scoreable traces (Trace 1 GPT-OSS, Trace 1 Llama, Trace 2, Trace 3) and logs the result as Feedback directly onto each trace_id. Trace 4 and 5 are excluded since they are rejection examples, not song recommendations.

In [0]:
from mlflow.genai.judges import make_judge

vibe_match_judge = make_judge(
    name="vibe_match_quality",
    instructions=(
        "Rate how well the songs in {{ outputs }} match the user's request in {{ inputs }}. "
        "Score from 1 to 5, where 1 means the songs do not match at all and 5 means they are a strong match."
    ),
    model=f"databricks:/{LLM_MODEL}",
    feedback_value_type=int,
)

print(f"Judge created: {vibe_match_judge.name}")

In [0]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

def score_and_log(trace_id, user_request, result):
    """
    Runs the judge on a trace's output and logs the score directly onto that trace_id.
    """
    songs_text = "\n".join([f"{s['track_name']} by {s['artists']}" for s in result["top_songs"]])
    
    judge_feedback = vibe_match_judge(
        inputs={"request": user_request},
        outputs={"songs": songs_text, "genre": result["genre"]}
    )
    
    mlflow.log_feedback(
        trace_id=trace_id,
        name=vibe_match_judge.name,
        value=judge_feedback.value,
        source=AssessmentSource(
            source_type=AssessmentSourceType.LLM_JUDGE,
            source_id=LLM_MODEL
        ),
        rationale=getattr(judge_feedback, "rationale", "")
    )
    
    return judge_feedback.value, getattr(judge_feedback, "rationale", "")

judge_records = []

score, rationale = score_and_log(trace1_id, user_request_1, trace1_result)
judge_records.append({"trace": "Trace 1 (GPT-OSS-120B)", "trace_id": trace1_id, "judge_score": score, "rationale": rationale})

score, rationale = score_and_log(trace1b_id, user_request_1, trace1b_result)
judge_records.append({"trace": "Trace 1 (Llama 3.3 70B)", "trace_id": trace1b_id, "judge_score": score, "rationale": rationale})

score, rationale = score_and_log(trace2_id, user_request_2, trace2_result)
judge_records.append({"trace": "Trace 2", "trace_id": trace2_id, "judge_score": score, "rationale": rationale})

score, rationale = score_and_log(trace3_id, user_request_3, trace3_result)
judge_records.append({"trace": "Trace 3", "trace_id": trace3_id, "judge_score": score, "rationale": rationale})

judge_df = pd.DataFrame(judge_records)

for _, row in judge_df.iterrows():
    print(f"{row['trace']} (score: {row['judge_score']})")
    print(f"  {row['rationale']}\n")

##### Commentary

## Human-in-the-Loop Evaluation

We review the same four traces the judge scored and logs its own rating directly onto each trace_id, using the same feedback name (vibe_match_quality) as the judge so both scores sit side by side on the same trace in the MLflow UI.

In [0]:
team_reviews = [
    {
        "trace": "Trace 1 (GPT-OSS-120B)",
        "trace_id": trace1_id,
        "team_score": 3,
        "team_rationale": "Reza Forte and Saci hit the gym energy, Soul Makossa is a real afrobeat classic but too laid back for a workout. Solid, not perfect."
    },
    {
        "trace": "Trace 1 (Llama 3.3 70B)",
        "trace_id": trace1b_id,
        "team_score": 1,
        "team_rationale": "Agree with the judge, and would actually go lower. None of these tracks read as afrobeat or gym-paced to an actual listener."
    },
    {
        "trace": "Trace 2",
        "trace_id": trace2_id,
        "team_score": 4,
        "team_rationale": "Agree with the judge. Deep-house picks genuinely fit a late night drive, and having a song literally called Drive is a nice touch."
    },
    {
        "trace": "Trace 3",
        "trace_id": trace3_id,
        "team_score": 2,
        "team_rationale": "Genre is correct but only 2 songs came back. A 2-song result barely counts as a playlist, even if both tracks individually fit the calm/study vibe."
    }
]

for review in team_reviews:
    mlflow.log_feedback(
        trace_id=review["trace_id"],
        name=vibe_match_judge.name,
        value=review["team_score"],
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="vibeml_team"
        ),
        rationale=review["team_rationale"]
    )

team_df = pd.DataFrame(team_reviews)

comparison_df = judge_df[["trace", "trace_id", "judge_score"]].merge(
    team_df[["trace", "team_score", "team_rationale"]], on="trace"
)
comparison_df["agreement"] = comparison_df["judge_score"] == comparison_df["team_score"]

comparison_df

### Commentary: Human-in-the-Loop Review

TODO:

## Overall Agent Performance Summary

This section ties together what the five traces and the evaluation showed about how the agent performs overall, and compares the two LLMs directly.

### Final Commentary

- TODO: 